## Summary

### ✅ What Random Forest Does
1. **Takes 40+ Features**: CPU util, memory, network, temps, cooling metrics, weather, temporal
2. **Creates 100 Decision Trees**: Each tree learns different patterns in the data
3. **Combines Predictions**: Averages all trees to reduce noise and overfitting
4. **Outputs PUE Prediction**: Single value between ~1.0 (ideal) and ~3.0 (inefficient)

### 🎯 Why Random Forest?

| Reason | Benefit for PUE Prediction |
|--------|---------------------------|
| **Handles Non-Linear Patterns** | Energy efficiency (PUE) doesn't scale linearly with CPU—it has thresholds, tipping points, and complex interactions |
| **Minimal Tuning Needed** | Works well "out of the box" vs neural networks that need extensive hyperparameter optimization |
| **Feature Importance Built-In** | Automatically tells us which factors matter most (e.g., avg_cpu > outdoor_temp > cooling_cop) |
| **Fast Inference** | Predicts in milliseconds—useful for real-time monitoring |
| **Robust to Outliers** | Won't break when encountering unusual workload patterns |
| **Good Baseline Model** | Establishes performance standard before trying more complex approaches |

### 🚀 Next Steps
- Now that we have PUE predictions, use them for:
  - **Phase 3a**: Carbon-aware scheduling (route workloads to low-PUE times)
  - **Phase 3b**: Cooling optimization (adjust setpoints based on predicted PUE)
  - **Phase 3c**: Check LSTM workload forecaster to predict spikes

In [4]:
print("\nKEY INSIGHTS:")
print("=" * 70)
print(f"\n1. TOP FEATURES DRIVING PUE:")
for i, (feat, imp) in enumerate(zip(top_features[:5], top_importances[:5]), 1):
    print(f"   {i}. {feat:35s} → {imp:.4f}")

print(f"\n2. MODEL GENERALIZATION:")
print(f"   • Training R²:   {train_metrics['R²']:.4f}")
print(f"   • Validation R²: {val_metrics['R²']:.4f}")
print(f"   • Test R²:       {test_metrics['R²']:.4f}")
if train_metrics['R²'] - test_metrics['R²'] < 0.1:
    print(f"   ✓ Good generalization - model not overfitted!")
else:
    print(f"   ⚠ Some overfitting detected (gap > 0.1)")

print(f"\n3. TYPICAL PREDICTION ERROR:")
print(f"   • Mean Absolute Error: {test_metrics['MAE']:.4f} PUE points")
print(f"   • RMSE:                {test_metrics['RMSE']:.4f} PUE points")
print(f"   • Real world example: With MAE={test_metrics['MAE']:.2f}, a prediction")
print(f"     of PUE=2.0 could be off by ±{test_metrics['MAE']:.2f}")

print(f"\n4. WHY RANDOM FOREST WORKS HERE:")
print(f"   ✓ Captures non-linear relationships (CPU→Power→Cooling)")
print(f"   ✓ Automatically ranks feature importance")
print(f"   ✓ Fast inference for real-time predictions")
print(f"   ✓ No gradient descent = stable training")
print(f"   ✓ Works with 40+ diverse features")
print("=" * 70)


KEY INSIGHTS:

1. TOP FEATURES DRIVING PUE:


NameError: name 'top_features' is not defined

In [ ]:
# Get feature importances
feature_importances = rf_model.feature_importances_
top_n = 15

# Sort by importance
top_indices = np.argsort(feature_importances)[::-1][:top_n]
top_features = [feature_names[i] for i in top_indices]
top_importances = feature_importances[top_indices]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Top 15 Feature Importances
colors = plt.cm.viridis(np.linspace(0, 1, top_n))
axes[0].barh(range(top_n), top_importances, color=colors)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_features)
axes[0].set_xlabel('Importance Score', fontsize=12, fontweight='bold')
axes[0].set_title('Top 15 Features Affecting PUE', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (feat, imp) in enumerate(zip(top_features, top_importances)):
    axes[0].text(imp, i, f' {imp:.4f}', va='center', fontsize=9)

# Plot 2: Feature Importance Distribution
axes[1].hist(feature_importances, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Importance Score', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Features', fontsize=12, fontweight='bold')
axes[1].set_title('Distribution of Feature Importance Across All Features', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/Users/kajalpatel/Data-center-energy-optimization/notebooks/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("Feature importance plot saved!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted (Test Set)
axes[0].scatter(y_test, y_test_pred, alpha=0.6, s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual PUE', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted PUE', fontsize=12, fontweight='bold')
axes[0].set_title('Actual vs Predicted PUE (Test Set)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals (Errors)
residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, alpha=0.6, s=50, color='orange')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted PUE', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Residuals (Actual - Predicted)', fontsize=12, fontweight='bold')
axes[1].set_title('Prediction Errors', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/kajalpatel/Data-center-energy-optimization/notebooks/pue_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("Plot saved!")

In [ ]:
def evaluate_model(y_true, y_pred, split_name):
    """Calculate evaluation metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'split': split_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }

# Evaluate model on all splits
train_metrics = evaluate_model(y_train, y_train_pred, 'Training')
val_metrics = evaluate_model(y_val, y_val_pred, 'Validation')
test_metrics = evaluate_model(y_test, y_test_pred, 'Test')

# Display results
print("Model Performance Metrics:")
print("=" * 70)
metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])
print(metrics_df.to_string(index=False))

print("\n" + "=" * 70)
print(f"TEST SET PERFORMANCE:")
print(f"  • R² Score:  {test_metrics['R²']:.4f}  (how much variance explained)")
print(f"  • RMSE:      {test_metrics['RMSE']:.4f}  (typical prediction error)")
print(f"  • MAE:       {test_metrics['MAE']:.4f}  (average absolute error)")
print("=" * 70)

In [ ]:
print("Making Predictions...")
print("=" * 70)

# Predict on all splits
y_train_pred = rf_model.predict(X_train)
y_val_pred = rf_model.predict(X_val)
y_test_pred = rf_model.predict(X_test)

print(f"✓ Predictions made on all splits")
print(f"\nSample Predictions vs Actual:")
for i in range(5):
    print(f"  Test[{i}]: Predicted={y_test_pred[i]:.4f}, Actual={y_test[i]:.4f}, Error={abs(y_test_pred[i]-y_test[i]):.4f}")


In [ ]:
print("Training Random Forest Model...")
print("=" * 70)

# Initialize model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

# Train
rf_model.fit(X_train, y_train)

print("✓ Model trained on {0} samples with {1} features".format(
    len(X_train), X_train.shape[1]
))
print(f"✓ Model has {rf_model.n_estimators} decision trees")

### Random Forest Architecture

**Hyperparameters:**
- `n_estimators=100`: 100 decision trees in the ensemble
- `max_depth=15`: Prevents trees from getting too deep (avoids overfitting)
- `min_samples_split=5`: Needs at least 5 samples to split a node
- `random_state=42`: Reproducible results

**Why these values?**
- 100 trees = balance between accuracy and speed
- max_depth=15 = prevents memorizing noise
- min_samples_split=5 = regularization to avoid overfitting

In [ ]:
# Split data: 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.176 ≈ 15% of original
)

print("Data Split:")
print(f"Training:   {len(X_train):4d} samples ({len(X_train)/len(X_scaled)*100:5.1f}%)")
print(f"Validation: {len(X_val):4d} samples ({len(X_val)/len(X_scaled)*100:5.1f}%)")
print(f"Test:       {len(X_test):4d} samples ({len(X_test)/len(X_scaled)*100:5.1f}%)")
print(f"Total:      {len(X_scaled):4d} samples (100.0%)")

print(f"\nTrain-Test-Val PUE ranges:")
print(f"  Train: {y_train.min():.3f} - {y_train.max():.3f} (mean: {y_train.mean():.3f})")
print(f"  Val:   {y_val.min():.3f} - {y_val.max():.3f} (mean: {y_val.mean():.3f})")
print(f"  Test:  {y_test.min():.3f} - {y_test.max():.3f} (mean: {y_test.mean():.3f})")

In [ ]:
# Prepare features and target
y = df['pue'].values  # Target: PUE
X = df.drop(columns=['timestamp', 'pue']).values  # Features
feature_names = df.drop(columns=['timestamp', 'pue']).columns.tolist()

print(f"Features: {len(feature_names)}")
print(f"Feature list: {feature_names[:10]}... (showing 10 of {len(feature_names)})")
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

# Normalize features
print("\nNormalizing features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"✓ Features normalized (mean≈0, std≈1)")

In [ ]:
print("Dataset Overview:")
print("=" * 70)
print(df.info())
print("\nFirst 5 rows:")
print(df.head())
print("\nTarget Variable (PUE) Statistics:")
print(df['pue'].describe())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
from data_generation.synthetic_generator import StreamDataGenerator, DataCenterConfig
from features.feature_engineer import FeatureEngineer

# Generate synthetic training data
print("Generating synthetic data (1 week = 168 hours)...")
config = DataCenterConfig(num_servers=50, num_racks=5, num_chillers=2, num_crac_units=4)
generator = StreamDataGenerator(config)
feature_engineer = FeatureEngineer()

# Generate 168 hourly batches (1 week)
start_time = datetime.now(timezone.utc) - timedelta(hours=168)
for i, batch in enumerate(generator.stream_generator(
    start_time=start_time,
    num_batches=168,
    interval_seconds=3600
)):
    feature_engineer.process_batch(batch)
    if (i + 1) % 24 == 0:
        print(f"  Generated {i + 1}/168 batches...")

# Get feature dataframe
df = feature_engineer.get_feature_dataframe()
print(f"\n✓ Dataset created: {len(df)} rows × {df.shape[1]} columns")

In [ ]:
import sys
sys.path.insert(0, '/Users/kajalpatel/Data-center-energy-optimization/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta, timezone

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")

# Random Forest PUE Predictor 🌲

## What is PUE?
**Power Usage Effectiveness (PUE)** measures data center energy efficiency:
- **PUE = Total Facility Power / IT Equipment Power**
- **Lower = Better** (ideal is 1.0, meaning all power goes to IT)
- Real data centers typically range 1.5 - 2.5

## Why Random Forest?

| Aspect | Why Random Forest Works |
|--------|------------------------|
| **Non-linear Data** | PUE has complex relationships with many features (CPU, temps, weather, etc.) |
| **Minimal Tuning** | Works well out-of-the-box with sensible defaults |
| **Feature Importance** | Built-in feature importance tells us what drives efficiency |
| **Fast Inference** | No neural network overhead - perfect for real-time predictions |
| **Robust** | Handles outliers and missing patterns well |
| **Good Baseline** | Provides foundation before moving to deeper models |

## Model Overview

```
Input: 40+ engineered features
├─ Workload (CPU, Memory, Disk)
├─ Cooling (Chiller efficiency, temps)
├─ Temporal (Hour, day of week)
└─ Weather (Outdoor temp, humidity)
            ↓
       [Random Forest]
       100 Decision Trees
            ↓
     Output: PUE Prediction (1.0-3.0)
```
